# MarketPulse - Entrega 2: datos y preparación

Pipeline diario **Bronze / Raw -> Silver -> Gold**, con datos reales de Yahoo Finance y transformaciones visibles en este notebook. No se entrena ningún modelo.

**Cómo ejecutarlo:** instala `requirements.txt`, abre este archivo en un editor de notebooks, selecciona el Python de `.venv` y ejecuta todas las celdas en orden. La primera ejecución descarga los datos; las posteriores verifican y reutilizan Bronze sin conexión. Para otra adquisición elige un `DATA_DIR` nuevo en la configuración.

El documento [Entrega 2](../docs/entregas/02_datos_y_preparacion.md) desarrolla los cinco apartados del PDF; [data/README.md](../data/README.md) explica el acceso y las salidas. El notebook contiene todo el código del pipeline y no depende de un script externo.

## 1. Entorno y configuración

Una observación es un activo en una sesión diaria. El periodo va de 2022-01-01 (inclusive) a 2025-01-01 (exclusive). Los precios se expresan en USD; los volúmenes, en unidades negociadas. Las ventanas de referencia incluyen 20 observaciones anteriores del mismo activo. Se requieren al menos 22 observaciones válidas por activo para obtener la primera fila Gold.

Yahoo Finance se consulta mediante yfinance, una biblioteca no oficial. Su licencia no licencia las cotizaciones: véase el [aviso de uso personal](https://pypi.org/project/yfinance/). No se recopilan datos personales ni se publican los CSV en Git.

In [1]:
from pathlib import Path
from types import SimpleNamespace
from datetime import date, datetime, timezone
from importlib.metadata import version
import hashlib
import json
import platform

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'docs/entregas/02_datos_y_preparacion.md').is_file()), None)
if ROOT is None:
    raise RuntimeError('Ejecuta el notebook desde la raíz del repo o la carpeta notebooks.')
NOTEBOOK_PATH = ROOT / 'notebooks/02_marketpulse_datos_y_preparacion.ipynb'
DATA_DIR = ROOT / 'data'
SYMBOLS = ['AAPL', 'MSFT', 'NVDA', 'AMZN', 'META', 'GOOGL', 'SPY', 'QQQ']
START = '2022-01-01'
END = '2025-01-01'
WINDOW = 20
RAW_COLUMNS = ['Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']
FEATURES = ['daily_return', 'intraday_range_pct', 'volume_ma_20', 'relative_volume',
            'volatility_20', 'rolling_high_20', 'rolling_low_20',
            'distance_to_high_20', 'distance_to_low_20']
GOLD_COLUMNS = ['date', 'symbol', 'close', 'adj_close', 'volume'] + FEATURES

SYMBOLS = list(dict.fromkeys(s.strip().upper() for s in SYMBOLS))
if not SYMBOLS or any(not s or not all(c.isalnum() or c in '.-^=' for c in s) for s in SYMBOLS):
    raise ValueError('Tickers inválidos; no pueden incluir separadores de carpetas.')
if date.fromisoformat(START) >= date.fromisoformat(END):
    raise ValueError('START debe preceder a END.')
args = SimpleNamespace(symbols=SYMBOLS, start=START, end=END, data_dir=DATA_DIR,
                       reuse_bronze=(DATA_DIR / 'bronze/manifest.json').is_file())
display(pd.DataFrame([{'inicio': START, 'fin_exclusivo': END, 'activos': len(SYMBOLS),
                       'ventana': WINDOW, 'reutiliza_bronze': args.reuse_bronze}]))
display(pd.DataFrame([{'paquete': p, 'version': version(p)}
                      for p in ['pandas', 'numpy', 'yfinance']]))

,inicio,fin_exclusivo,activos,ventana,reutiliza_bronze
0,2022-01-01,2025-01-01,8,20,True


,paquete,version
0,pandas,3.0.1
1,numpy,2.3.5
2,yfinance,0.2.66


## 2. Bronze / Raw: adquisición y trazabilidad

Se conserva un CSV por ticker con los valores recibidos de la biblioteca, sin limpiar ni calcular variables. El índice de sesión se serializa como `Date`. El manifiesto registra fuente, parámetros, fecha UTC, versiones, filas y SHA-256 de cada CSV. No es la respuesta HTTP original, sino la tabla devuelta por yfinance.

Si existe un snapshot se comprueban configuración y hashes antes de reutilizarlo. Se exige obtener todos los activos y no se sobrescribe un Bronze existente. Otra descarga puede variar debido a revisiones del proveedor.

In [2]:
def digest(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def acquire_bronze(args, directory: Path) -> pd.DataFrame:
    """Persist the uncleaned provider table, or verify and replay that snapshot."""
    manifest_path = directory / 'manifest.json'
    config = {'symbols': args.symbols, 'start': args.start, 'end': args.end,
              'interval': '1d', 'auto_adjust': False, 'window': WINDOW}
    if args.reuse_bronze:
        manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
        if manifest['config'] != config:
            raise ValueError('Bronze configuration differs; use the same symbols/start/end.')
        for symbol in args.symbols:
            path = directory / f'{symbol}.csv'
            if digest(path) != manifest['files'][symbol]['sha256']:
                raise ValueError(f'Bronze checksum mismatch: {symbol}')
    else:
        if directory.exists():
            raise ValueError('Bronze already exists. Use --reuse-bronze or another --data-dir.')
        import yfinance as yf
        frames = {}
        for symbol in args.symbols:
            print(f'Downloading daily bars: {symbol}', flush=True)
            frame = yf.download(symbol, start=args.start, end=args.end, interval='1d',
                                auto_adjust=False, actions=False, repair=False, keepna=True,
                                progress=False, threads=False, multi_level_index=False,
                                timeout=30)
            if frame is None or frame.empty:
                raise RuntimeError(f'No data for {symbol}; no partial dataset published.')
            frame = frame.reset_index()
            missing = set(RAW_COLUMNS) - set(frame.columns)
            if missing:
                raise ValueError(f'{symbol}: missing provider columns {sorted(missing)}')
            frames[symbol] = frame
        directory.mkdir(parents=True)
        manifest = {'source': 'Yahoo Finance via yfinance', 'config': config,
                    'retrieved_at_utc': datetime.now(timezone.utc).isoformat(),
                    'versions': {'python': platform.python_version(),
                                 **{p: version(p) for p in ['pandas', 'numpy', 'yfinance']}},
                    'files': {}}
        for symbol, frame in frames.items():
            path = directory / f'{symbol}.csv'
            frame.to_csv(path, index=False, float_format='%.17g', lineterminator='\n')
            manifest['files'][symbol] = {'rows': len(frame), 'sha256': digest(path)}
        manifest_path.write_text(json.dumps(manifest, indent=2) + '\n', encoding='utf-8')
    frames = []
    for symbol in args.symbols:
        frame = pd.read_csv(directory / f'{symbol}.csv', float_precision='round_trip')
        frame['Symbol'] = symbol
        frames.append(frame)
    return pd.concat(frames, ignore_index=True)

In [3]:
bronze = acquire_bronze(args, DATA_DIR / 'bronze')
manifest = json.loads((DATA_DIR / 'bronze/manifest.json').read_text(encoding='utf-8'))
print(f"Bronze: {len(bronze):,} filas; {bronze['Symbol'].nunique()} activos.")
print('Origen:', manifest['source'])
print('Adquisición UTC:', manifest['retrieved_at_utc'])
display(bronze.head(5))
display(bronze.groupby('Symbol').size().rename('filas_bronze').to_frame())

Bronze: 6,024 filas; 8 activos.
Origen: Yahoo Finance via yfinance
Adquisición UTC: 2026-10-07T15:51:59.372202+00:00


,Date,Adj Close,Close,High,Low,Open,Volume,Symbol
0,2022-01-03,177.786377,182.009995,182.880005,177.710007,177.830002,104487900,AAPL
1,2022-01-04,175.530045,179.699997,182.940002,179.119995,182.630005,99310400,AAPL
2,2022-01-05,170.860916,174.919998,180.169998,174.639999,179.610001,94537600,AAPL
3,2022-01-06,168.008682,172.000000,175.300003,171.639999,172.699997,96904000,AAPL
4,2022-01-07,168.174728,172.169998,174.139999,171.029999,172.889999,86709100,AAPL


,filas_bronze
Symbol,
AAPL,753
AMZN,753
GOOGL,753
META,753
MSFT,753
NVDA,753
QQQ,753
SPY,753


## 3. Silver: limpieza y variables históricas

Las transformaciones están en la siguiente celda: nombres y tipos uniformes, orden por activo/fecha, eliminación de duplicados exactos y descarte de nulos, infinitos, precios no positivos, volumen negativo/fraccionario y OHLC incoherente. Un duplicado contradictorio detiene el proceso. No se imputan huecos.

La rentabilidad utiliza el cierre ajustado. Máximos y mínimos se ajustan con `adj_close / close` para compararlos con ese cierre. Volumen medio, volatilidad y extremos móviles usan exclusivamente las 20 observaciones de **t-1 hacia atrás** por activo. Las variables de t se conocen al cierre de t. La volatilidad es muestral, sin anualizar. Si faltan sesiones, la rentabilidad entre observaciones puede abarcar más de una sesión: revisa esa cobertura antes de ML.

In [4]:
def build_silver(bronze: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    silver = bronze.copy()
    silver.columns = [str(c).strip().lower().replace(' ', '_') for c in silver.columns]
    required = {'date', 'symbol', 'open', 'high', 'low', 'close', 'adj_close', 'volume'}
    if missing := required - set(silver.columns):
        raise ValueError(f'Missing columns: {sorted(missing)}')
    report = {'bronze_rows': len(silver)}
    silver['date'] = pd.to_datetime(silver['date'], errors='coerce', format='mixed')
    # Daily source dates are session labels: preserve their calendar date, not UTC shifts.
    if isinstance(silver['date'].dtype, pd.DatetimeTZDtype):
        silver['date'] = silver['date'].dt.tz_localize(None)
    silver['date'] = silver['date'].dt.normalize()
    silver['symbol'] = silver['symbol'].astype('string').str.strip().str.upper()
    numeric = ['open', 'high', 'low', 'close', 'adj_close', 'volume']
    for column in numeric:
        silver[column] = pd.to_numeric(silver[column], errors='coerce')
    valid = silver['date'].notna() & silver['symbol'].notna() & silver['symbol'].ne('')
    valid &= pd.Series(np.isfinite(silver[numeric].to_numpy(dtype=float)).all(axis=1), index=silver.index)
    valid &= silver[['open', 'high', 'low', 'close', 'adj_close']].gt(0).all(axis=1)
    valid &= silver['volume'].ge(0) & silver['volume'].mod(1).eq(0)
    valid &= silver['high'].ge(silver[['open', 'close', 'low']].max(axis=1))
    valid &= silver['low'].le(silver[['open', 'close', 'high']].min(axis=1))
    report['invalid_rows_removed'] = int((~valid).sum())
    silver = silver.loc[valid].copy()
    count = len(silver)
    silver = silver.drop_duplicates()
    report['exact_duplicates_removed'] = count - len(silver)
    if silver.duplicated(['symbol', 'date']).any():
        raise ValueError('Conflicting bars for the same symbol/date; inspect Bronze.')
    silver = silver.sort_values(['symbol', 'date']).reset_index(drop=True)
    silver['volume'] = silver['volume'].astype('int64')
    if silver.empty:
        raise ValueError('No valid rows remain after cleaning.')
    frames = []
    for _, group in silver.groupby('symbol', sort=True):
        group = group.copy()
        factor = group['adj_close'] / group['close']
        adjusted_high = group['high'] * factor
        adjusted_low = group['low'] * factor
        group['daily_return'] = group['adj_close'].pct_change(fill_method=None)
        group['intraday_range_pct'] = (group['high'] - group['low']) / group['close']
        # Reference windows end at t-1, so today's observation cannot dilute its baseline.
        group['volume_ma_20'] = group['volume'].shift(1).rolling(WINDOW, min_periods=WINDOW).mean()
        group['relative_volume'] = group['volume'] / group['volume_ma_20'].replace(0, np.nan)
        group['volatility_20'] = group['daily_return'].shift(1).rolling(WINDOW, min_periods=WINDOW).std(ddof=1)
        group['rolling_high_20'] = adjusted_high.shift(1).rolling(WINDOW, min_periods=WINDOW).max()
        group['rolling_low_20'] = adjusted_low.shift(1).rolling(WINDOW, min_periods=WINDOW).min()
        group['distance_to_high_20'] = group['adj_close'] / group['rolling_high_20'] - 1
        group['distance_to_low_20'] = group['adj_close'] / group['rolling_low_20'] - 1
        frames.append(group)
    silver = pd.concat(frames, ignore_index=True).replace([np.inf, -np.inf], np.nan)
    report['silver_rows'] = len(silver)
    return silver, report

In [5]:
silver, quality = build_silver(bronze)
display(pd.DataFrame([quality]))
display(silver[['date', 'symbol', 'adj_close', 'daily_return',
                'volume_ma_20', 'relative_volume', 'volatility_20']].head(25))

,bronze_rows,invalid_rows_removed,exact_duplicates_removed,silver_rows
0,6024,0,0,6024


,date,symbol,adj_close,daily_return,volume_ma_20,relative_volume,volatility_20
0,2022-01-03,AAPL,177.786377,NaN,NaN,NaN,NaN
1,2022-01-04,AAPL,175.530045,-0.012691,NaN,NaN,NaN
2,2022-01-05,AAPL,170.860916,-0.026600,NaN,NaN,NaN
3,2022-01-06,AAPL,168.008682,-0.016693,NaN,NaN,NaN
4,2022-01-07,AAPL,168.174728,0.000988,NaN,NaN,NaN
5,2022-01-10,AAPL,168.194275,0.000116,NaN,NaN,NaN
6,2022-01-11,AAPL,171.017212,0.016784,NaN,NaN,NaN
7,2022-01-12,AAPL,171.456802,0.002570,NaN,NaN,NaN
8,2022-01-13,AAPL,168.194275,-0.019028,NaN,NaN,NaN
9,2022-01-14,AAPL,169.053833,0.005111,NaN,NaN,NaN


## 4. Gold: dataset para la próxima fase

Cada fila representa un activo en una sesión, con sus variables completas. Se eliminan el calentamiento inicial y otras filas con variables indefinidas; no se rellenan con cero. Gold servirá para comparar baselines y métodos de anomalías en una entrega posterior. No hay etiquetas inventadas ni entrenamiento.

| Variables | Significado |
|---|---|
| `date`, `symbol` | Clave única de sesión y activo |
| `close`, `adj_close`, `volume` | Cierre, cierre ajustado y actividad |
| `daily_return` | Rentabilidad ajustada entre observaciones |
| `intraday_range_pct` | `(high - low) / close` |
| `volume_ma_20`, `relative_volume` | Media previa y volumen actual / media previa |
| `volatility_20` | Desviación típica muestral de 20 rentabilidades previas |
| `rolling_high_20`, `rolling_low_20` | Extremos ajustados previos |
| `distance_to_high_20`, `distance_to_low_20` | Cierre ajustado / extremo previo - 1 |

Las rentabilidades, distancias, rangos y volatilidad son fracciones (0.02 = 2 %). El volumen relativo es un múltiplo (2 = dos veces la media).

In [6]:
def build_gold(silver: pd.DataFrame) -> pd.DataFrame:
    gold = silver[GOLD_COLUMNS].dropna().copy()
    if gold.empty:
        raise ValueError('Gold is empty: provide more than 21 valid daily sessions per symbol.')
    return gold.sort_values(['symbol', 'date']).reset_index(drop=True)

In [7]:
gold = build_gold(silver)
assert not gold.isna().any().any(), 'Gold tiene nulos'
assert np.isfinite(gold.select_dtypes('number')).all().all(), 'Gold tiene infinitos'
assert not gold.duplicated(['symbol', 'date']).any(), 'Gold tiene claves duplicadas'
assert set(gold['symbol']) == set(SYMBOLS), 'Faltan activos en Gold'
print(f'Gold validado: {len(gold):,} filas; {gold.symbol.nunique()} activos.')
display(gold.head(5))
display(gold.groupby('symbol').agg(filas=('date', 'size'),
                                 primera_sesion=('date', 'min'),
                                 ultima_sesion=('date', 'max')))

Gold validado: 5,856 filas; 8 activos.


,date,symbol,close,adj_close,volume,daily_return,intraday_range_pct,volume_ma_20,relative_volume,volatility_20,rolling_high_20,rolling_low_20,distance_to_high_20,distance_to_low_20
0,2022-02-02,AAPL,175.839996,171.759537,84914300,0.007044,0.014502,104508600.0,0.812510,0.021199,178.694865,151.110135,-0.038811,0.136651
1,2022-02-03,AAPL,172.899994,168.887787,89418100,-0.016720,0.023829,103788795.0,0.861539,0.021128,175.989088,151.110135,-0.040351,0.117647
2,2022-02-04,AAPL,172.389999,168.604172,82465400,-0.001679,0.019839,103532820.0,0.796515,0.020604,173.068508,151.110135,-0.025795,0.115770
3,2022-02-07,AAPL,171.660004,167.890228,77251200,-0.004234,0.017476,102810890.0,0.751391,0.020249,173.068508,151.110135,-0.029920,0.111045
4,2022-02-08,AAPL,174.830002,170.990601,74829200,0.018467,0.022422,102337995.0,0.731197,0.020274,173.068508,151.110135,-0.012006,0.131563


,filas,primera_sesion,ultima_sesion
symbol,,,
AAPL,732,2022-02-02,2024-12-31
AMZN,732,2022-02-02,2024-12-31
GOOGL,732,2022-02-02,2024-12-31
META,732,2022-02-02,2024-12-31
MSFT,732,2022-02-02,2024-12-31
NVDA,732,2022-02-02,2024-12-31
QQQ,732,2022-02-02,2024-12-31
SPY,732,2022-02-02,2024-12-31


## 5. Guardar las capas y comprobar la reproducción

Se guardan Silver, Gold y un informe de calidad con descartes, cobertura y hashes. La huella del pipeline se calcula sobre el código de las celdas, excluyendo sus resultados. Bronze permanece intacto y los CSV generados se excluyen de Git.

La última celda vuelve a procesar el snapshot sin descargar y exige igualdad de los datos y del archivo Gold serializado. Mantén las mismas versiones del entorno para repetir exactamente el resultado.

In [8]:
def pipeline_digest(path):
    notebook = json.loads(path.read_text(encoding='utf-8'))
    sources = []
    for cell in notebook['cells']:
        if cell['cell_type'] == 'code':
            source = cell['source']
            sources.append(''.join(source) if isinstance(source, list) else source)
    return hashlib.sha256('\n'.join(sources).encode('utf-8')).hexdigest()


def save_outputs(args, silver, gold, report):
    missing = set(args.symbols) - set(gold['symbol'])
    if missing:
        raise ValueError(f'Symbols without usable Gold rows: {sorted(missing)}')
    # Validate all outputs before publishing Silver and Gold.
    for name, frame in [('silver', silver), ('gold', gold)]:
        directory = args.data_dir / name
        directory.mkdir(parents=True, exist_ok=True)
        frame.to_csv(directory / 'marketpulse.csv', index=False, date_format='%Y-%m-%d',
                     float_format='%.17g', lineterminator='\n')
    report['gold_rows'] = len(gold)
    report['warmup_or_undefined_feature_rows_removed'] = len(silver) - len(gold)
    report['per_symbol'] = {symbol: {'rows': len(group),
                                   'first_date': group['date'].min().strftime('%Y-%m-%d'),
                                   'last_date': group['date'].max().strftime('%Y-%m-%d')}
                            for symbol, group in gold.groupby('symbol')}
    report['gold_sha256'] = digest(args.data_dir / 'gold' / 'marketpulse.csv')
    report['bronze_manifest_sha256'] = digest(args.data_dir / 'bronze' / 'manifest.json')
    report['pipeline_sha256'] = pipeline_digest(NOTEBOOK_PATH)
    report['versions'] = {'python': platform.python_version(),
                          'pandas': version('pandas'), 'numpy': version('numpy')}
    (args.data_dir / 'gold' / 'quality_report.json').write_text(
        json.dumps(report, indent=2) + '\n', encoding='utf-8')


In [9]:
save_outputs(args, silver, gold, quality)
display(pd.DataFrame([{k: quality[k] for k in ['bronze_rows', 'silver_rows', 'gold_rows',
    'invalid_rows_removed', 'exact_duplicates_removed',
    'warmup_or_undefined_feature_rows_removed']}]))
print('Gold SHA-256:', quality['gold_sha256'])
print('Archivos: data/silver/marketpulse.csv, data/gold/marketpulse.csv y quality_report.json')

,bronze_rows,silver_rows,gold_rows,invalid_rows_removed,exact_duplicates_removed,warmup_or_undefined_feature_rows_removed
0,6024,6024,5856,0,0,168


Gold SHA-256: bd484aaf16a298f06935f6fe0fe37a08a5629577ae56ded31a13ecb446e41418
Archivos: data/silver/marketpulse.csv, data/gold/marketpulse.csv y quality_report.json


In [10]:
replay_args = SimpleNamespace(**{**vars(args), 'reuse_bronze': True})
replayed_bronze = acquire_bronze(replay_args, DATA_DIR / 'bronze')
replayed_silver, _ = build_silver(replayed_bronze)
replayed_gold = build_gold(replayed_silver)
pd.testing.assert_frame_equal(gold, replayed_gold)
replayed_csv = replayed_gold.to_csv(index=False, date_format='%Y-%m-%d',
                                   float_format='%.17g', lineterminator='\n')
assert hashlib.sha256(replayed_csv.encode('utf-8')).hexdigest() == quality['gold_sha256']
print('Reproducción sin red: datos idénticos y SHA-256 de Gold coincidente.')

Reproducción sin red: datos idénticos y SHA-256 de Gold coincidente.


## Riesgos y límites

Faltan noticias y eventos para explicar las anomalías. La selección de grandes acciones y ETF estadounidenses introduce sesgo de selección y supervivencia. Los datos diarios no capturan señales intradía y los ajustes retrospectivos no son necesariamente datos históricos disponibles en cada instante. Si falla Yahoo, se conserva Bronze y se puede adaptar una fuente diaria ajustada alternativa tras revisar sus condiciones. **No se ha entrenado ningún modelo.**